# Notebook 30 (simple): Pixel-Space Diffusion with a Transformer Bottleneck

---

## What This Notebook Covers

This is the **pixel-space** companion to the latent-diffusion notebook. Instead of compressing images into a VAE latent and diffusing there, here we diffuse **directly on low-resolution pixels** (64×64 LSUN bedrooms) — and introduce a new architectural idea: a **transformer bottleneck**. The U-Net's convolutional encoder and decoder are kept, but its middle section is replaced by a stack of full **transformer blocks** operating on the flattened bottleneck feature map. We will learn:

1. **Low-resolution pixel-space diffusion** — downsample 256×256 → 64×64 and diffuse in pixel space (no VAE), the cheap way to get a working image generator
2. **`LearnEmbSS`** — a learned per-position scale/shift, i.e. a learned positional embedding for the bottleneck sequence
3. **`EmbTransformerBlk`** — a canonical transformer block (pre-norm self-attention + MLP) with FiLM time conditioning bolted in
4. **`SimpleDiffusion`** — a hybrid architecture: convolutional down-path → **transformer bottleneck** → convolutional up-path
5. **Training and DDIM sampling** in pixel space, reusing `noisify`, `sample`, and `init_ddpm` from notebook 28

Everything diffusion-specific (`noisify`, `timestep_embedding`, `DownBlock`, `UpBlock`, `SelfAttention`, `sample`, `init_ddpm`) is imported from `miniai.diffusion` — the library we built in notebook 28. What is new here is the **transformer-in-the-middle** architecture.

> **A naming note, up front.** The source notebook's first cell is titled "Tiny Imagenet," but the data it actually downloads and trains on is **LSUN bedrooms** (the same `bedroom.tgz` as the latent notebook), downsampled to 64×64. Jeremy reused a template title; the content is bedrooms. I flag this so you are not confused when the data-loading cell fetches LSUN rather than ImageNet.

---

## Why This Variant?

There are two ways to make high-resolution diffusion affordable, and notebook 30 explores both:

- **The latent route** (the sibling `30_lsun_diffusion-latents` notebook): compress to a VAE latent, diffuse in the small space, decode. Best quality-per-compute, and the basis of Stable Diffusion.
- **The simple route (this notebook):** just diffuse at *low resolution* in pixel space — downsample to 64×64, generate, and (optionally) upsample later. No VAE to load or manage; the whole pipeline is one model. It is the most direct way to get a working image generator, and a clean setting to introduce a new architectural idea.

That new idea is the **transformer bottleneck.** In notebook 28 the U-Net was convolutions throughout, with self-attention *added* inside some resblocks. Here we go further: at the bottleneck — where the feature map is smallest (8×8) and a *global* operation is both most valuable and cheapest — we drop the convolutions entirely and run a stack of **pure transformer blocks** over the flattened 8×8 = 64-position sequence. The encoder and decoder stay convolutional (good at local, translation-equivariant features); the middle becomes a small Transformer (good at global, content-dependent mixing). This conv–transformer hybrid is a recurring, powerful pattern.

**Climate / EO bridge (a real one).** The "convolutional encoder → transformer core → convolutional decoder" shape is exactly the design of several modern **ML weather models**: a conv/patch stem reduces a high-resolution atmospheric field to a coarse grid of tokens, a Transformer models the *global* dynamics (teleconnections, long-range coupling) across those tokens, and a decoder projects back to the full grid (Pangu-Weather, FourCastNet-family, and ViT-hybrid downscalers all share this skeleton). Understanding `SimpleDiffusion` is understanding that skeleton in miniature.

---

## Prerequisites

You should be comfortable with:

- **Self-attention** (notebook 27) — `SelfAttention`, multi-head, the flatten-to-sequence move. Reused here with `transpose=False`.
- **The diffusion U-Net and its parts** (notebook 28) — `DownBlock`, `UpBlock`, `timestep_embedding`, FiLM `x*(1+scale)+shift` conditioning, `init_ddpm`, `noisify`, `sample`, `ddim_step`. All imported from `miniai.diffusion`.
- **The transformer block** — pre-norm, residual attention + residual MLP. The deep dive rebuilds it, but prior exposure helps.
- **DDPM/DDIM sampling** (notebooks 15–20).

---


## Google Colab Setup

Run the cells below **once** at the start of each Colab session. They mount Google Drive, set the working directory, install required packages, and clone the `miniai` library from the [fast.ai Part 2 course repo](https://github.com/fastai/course22p2).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Fast.AI_Colab')
print(os.getcwd())

In [ ]:
# Install required packages
!pip install -q fastcore fastai diffusers datasets torcheval accelerate einops timm

# Clone the fast.ai Part 2 course repo to get the miniai library
if not os.path.exists('course22p2'):
    !git clone https://github.com/fastai/course22p2.git

# Add miniai to the Python path
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'course22p2'))

# Verify miniai is accessible
try:
    import miniai
    print(f'miniai loaded successfully from: {miniai.__file__}')
except ImportError:
    print('ERROR: miniai not found. Check that course22p2 was cloned correctly.')

---

*The cells above are Colab-specific setup. The content below is the same as the source notebook (`30_lsun_diffusion-simple_explained.ipynb`). One cell was adjusted for the Colab environment (the `CUDA_VISIBLE_DEVICES` GPU selection); it is flagged with a note directly above it.*

> **Heads-up (resources).** This notebook downloads the multi-GB **LSUN bedroom** archive (~300k images) and trains a pixel-space diffusion model for 15 epochs at 64×64. On Colab: the download goes to ephemeral storage (point `path_data` at Drive to persist it), and 15 epochs on the full dataset will not finish in a free session — use a subset for a smoke test, or load a pretrained `models/lsun_diffusion-prog-64.pkl` and skip to sampling. The `torch.save(...)` cell writes to `models/` — create that folder (or point it at Drive) first.

---

# Part 1: Setup and Imports

Minimal setup. The one import that matters is `from miniai.diffusion import *`, which brings in the entire notebook-28 diffusion toolkit; this notebook only *adds* a transformer-bottleneck model on top.

---


> **Colab adjustment.** The original pins GPU index `2`; Colab exposes a single GPU at index `0`. Changed to `'0'` below (or delete this cell entirely on Colab).

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'

**What does the code above do?**

Pins training to GPU index 2 before any CUDA context exists. Set this to whatever device your machine has (`'0'` on a single-GPU box).


In [ ]:
from miniai.imports import *
from miniai.diffusion import *

from glob import glob

**What does the code above do?**

`from miniai.diffusion import *` pulls in everything we built in notebook 28: `noisify`, `abar`, `timestep_embedding`, `SelfAttention`, `DownBlock`, `UpBlock`, `EmbResBlock`, `lin`, `pre_conv`, `sample`, `init_ddpm`, and more. We reuse all of it; the only new code in this notebook is the transformer-bottleneck model. `glob` is for enumerating image files.


In [ ]:
torch.set_printoptions(precision=4, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'
mpl.rcParams['figure.dpi'] = 70

set_seed(42)
if fc.defaults.cpus>8: fc.defaults.cpus=8

**What does the code above do?**

The usual reproducibility and display boilerplate: readable printing, fixed seeds, modest DPI, and an 8-worker cap for data loading.


---

# Part 2: Data — LSUN Bedrooms at 64×64

We download LSUN bedrooms (same archive as the latent notebook), but process them very differently: crop to 256×256, then **average-pool by 4** down to **64×64**, and center pixels to `[-0.5, 0.5]`. Diffusing at 64×64 keeps pixel-space training tractable.

---


In [ ]:
path_data = Path('data')
path_data.mkdir(exist_ok=True)
path = path_data/'bedroom'

**What does the code above do?**

Sets up the local `data/` directory and the `data/bedroom` path where the extracted images live.


In [ ]:
url = 'https://s3.amazonaws.com/fast-ai-imageclas/bedroom.tgz'
if not path.exists():
    path_zip = fc.urlsave(url, path_data)
    shutil.unpack_archive('data/bedroom.tgz', 'data')

**What does the code above do?**

Downloads and unpacks the LSUN bedroom archive, once (guarded by `if not path.exists()`). This is the same ~300k-image dataset as the latent notebook — note it is **bedrooms**, despite the notebook's "Tiny Imagenet" title.


In [ ]:
bs = 256

**What does the code above do?**

Batch size 256 — larger than the latent notebook's, affordable because the 64×64 images are small.


In [ ]:
def to_img(f): return (read_image(f, mode=ImageReadMode.RGB)/255-0.5)

**What does the code above do?**

Reads an image as 3-channel RGB, scales to `[0, 1]`, and **subtracts 0.5** to center pixels to `[-0.5, 0.5]`. Centering matters for diffusion: the forward process adds zero-mean Gaussian noise, so zero-centered data keeps signal and noise symmetric around 0. (At sampling time we add 0.5 back to recover displayable `[0, 1]` pixels.)


In [ ]:
class ImagesDS:
    def __init__(self, spec):
        self.path = Path(path)
        self.files = glob(str(spec), recursive=True)
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        im = to_img(self.files[i])
        im = im[:, :256,:256]
        im = F.avg_pool2d(im, 4)
        return (im,)

**What does the code above do?**

The image dataset. `__getitem__` reads an image, crops to the top-left 256×256, and then **`F.avg_pool2d(im, 4)`** downsamples it 4× to **64×64** by averaging each 4×4 block. Average-pooling (rather than a strided crop or nearest-neighbor) is a cheap anti-aliased downsample — it smooths before subsampling, avoiding aliasing artifacts. It returns a 1-tuple `(im,)` because the `collate_ddpm` below indexes `[0]` to get the image (there are no labels — unconditional generation).


In [ ]:
tds = ImagesDS(path/'[1-9]'/f'**/*.jpg')
vds = ImagesDS(path/'0'/f'**/*.jpg')

**What does the code above do?**

Splits train/validation by top-level directory name: `tds` uses folders `1`–`9`, `vds` uses folder `0`. LSUN's images are sharded into numbered subdirectories, so this is a convenient ~90/10 split by shard (the `[1-9]` glob matches nine of the ten shards, `0` the tenth).


In [ ]:
def collate_ddpm(b): return noisify(default_collate(b)[0])

**What does the code above do?**

The diffusion collate: collate a batch, take `[0]` (the image out of the 1-tuple), and run notebook-28's `noisify` on it — which picks a random noise level per image and returns `((xₜ, t), ε)`. The model's job is to predict the noise `ε` from the noised image `xₜ` and the level `t`, exactly as in notebook 28, but now on 64×64 RGB pixels instead of latents.


In [ ]:
dls = DataLoaders(*get_dls(tds, vds, bs=bs, num_workers=fc.defaults.cpus, collate_fn=collate_ddpm))

**What does the code above do?**

Builds train/valid `DataLoaders` over the two splits, using `collate_ddpm` so every batch arrives already noisified into `((xₜ, t), ε)` — the format `SimpleDiffusion` expects.


---

# Part 3: A Learned Positional Embedding for the Bottleneck

The transformer bottleneck will treat the 8×8 bottleneck feature map as a **sequence of 64 positions**. But attention is *permutation-invariant* — it has no built-in notion of *where* each position is. `LearnEmbSS` supplies that missing position information as a learned per-position scale and shift.

---


In [ ]:
class LearnEmbSS(nn.Module):
    def __init__(self, sz, ni):
        super().__init__()
        self.scale = nn.Parameter(torch.zeros(sz, ni))
        self.shift = nn.Parameter(torch.zeros(sz, ni))

    def forward(self, x): return x*self.scale + self.shift

**What does the code above do?**

A tiny module holding two learnable tensors of shape `(sz, ni)` — a per-position `scale` and `shift` — and applying `x*scale + shift`. Here `sz=64` (the 8×8 bottleneck positions) and `ni` is the channel count. The deep dive explains why this is a *positional* embedding and why it is initialized to zero.


## Deep Dive: `LearnEmbSS` as a Learned Positional Embedding

### The problem: attention doesn't know *where*

When we flatten the 8×8 bottleneck feature map into a sequence of 64 position-vectors and feed it to self-attention, something is silently lost: **attention treats its input as a *set*, not a *sequence*.** Permuting the 64 positions and permuting them back leaves the attention output identical — the mechanism computes all-pairs comparisons with no reference to *which* position is where. For a bottleneck derived from a 2D image, that is a real loss: position (0,0) is the top-left corner and (7,7) the bottom-right, and the model should be able to use that spatial layout.

The standard fix is a **positional embedding**: add (or otherwise inject) a per-position learned vector so each position carries a unique, learnable "this is where I am" signal that attention *can* see.

### What `LearnEmbSS` does

`LearnEmbSS(64, ni)` holds two learnable `(64, ni)` tensors and computes, per position `p`:

$$\text{out}_p = x_p \odot \text{scale}_p + \text{shift}_p.$$

- **`shift_p`** is a classic additive positional embedding: a distinct learned vector added to each of the 64 positions. This is what breaks the permutation symmetry — after adding `shift`, position 0 and position 63 are distinguishable even if their input features were identical.
- **`scale_p`** is a learned per-position, per-channel *gain* — it additionally lets the model rescale features differently by location (a mild generalization of a plain additive embedding).

So `LearnEmbSS` is a **learned positional embedding with both an additive and a multiplicative part**, applied once to the sequence just before the transformer stack.

### Why initialize to zero?

Both `scale` and `shift` start as `torch.zeros`. At initialization, then, `out = x*0 + 0 = 0`... which would wipe out the features entirely — so why is that the right choice? Look at where it is used (next section): the result is fed into transformer blocks whose *own* residual structure and zero-friendly init mean the network starts in a stable regime, and the positional parameters **grow away from zero as training demands position information**. Starting the positional signal at zero means it does not disrupt the carefully-initialized feature statistics at step one; the model *learns how much* positional signal to inject rather than being forced to cope with a large random one from the start. This is the same "make new machinery start as a no-op and let it learn its strength" philosophy as the `1+scale` FiLM trick and `init_ddpm`.

> **Subtlety worth noting.** Because `scale` starts at 0, the *first* forward pass multiplies features by 0 at the bottleneck. In practice this is fine: gradients still flow to `scale`/`shift` (their gradients depend on `x` and the upstream signal), the residual transformer blocks downstream keep the pathway alive, and within a few steps the parameters move off zero. It is an intentional soft start, not a bug.


---

# Part 4: The Transformer Block

Now the core new component: a full transformer block with time conditioning. First the small MLP it uses, then the block itself.

---


In [ ]:
def _mlp(ni, nh):
    return nn.Sequential(nn.Linear(ni,nh), nn.GELU(), nn.LayerNorm(nh), nn.Linear(nh,ni))

**What does the code above do?**

The transformer's **feed-forward network (FFN)**: `Linear(ni→nh) → GELU → LayerNorm → Linear(nh→ni)`. It expands to a wider hidden size `nh` (here `ni*4`, the conventional 4× expansion), applies a nonlinearity, and projects back to `ni`. This per-position MLP is where the transformer does its position-wise nonlinear processing, complementing attention's cross-position mixing. (`GELU` — Gaussian Error Linear Unit — is the standard transformer activation.)


In [ ]:
class EmbTransformerBlk(nn.Module):
    def __init__(self, n_emb, ni, attn_chans=8):
        super().__init__()
        self.attn = SelfAttention(ni, attn_chans=attn_chans, transpose=False)
        self.mlp = _mlp(ni, ni*4)
        self.nrm1 = nn.LayerNorm(ni)
        self.nrm2 = nn.LayerNorm(ni)
        self.emb_proj = nn.Linear(n_emb, ni*2)

    def forward(self, x, t):
        emb = self.emb_proj(F.silu(t))[:, None]
        scale,shift = torch.chunk(emb, 2, dim=2)
        x = x + self.attn(self.nrm1(x))
        x = x*(1+scale) + shift
        return x + self.mlp(self.nrm2(x))

**What does the code above do?**

A transformer block with FiLM time conditioning. The deep dive breaks down each line; in brief: pre-norm residual self-attention, then FiLM modulation from the time embedding, then a pre-norm residual MLP.


## Deep Dive: `EmbTransformerBlk` — a Transformer Block with Time Conditioning

This is the canonical **pre-norm transformer block** you would find in any modern Transformer, with one addition for diffusion: it is modulated by the timestep embedding via FiLM. Let's read it against the standard template.

### The standard pre-norm transformer block

A transformer block is two residual sub-layers:

$$x \leftarrow x + \text{Attention}(\text{LN}(x)), \qquad x \leftarrow x + \text{MLP}(\text{LN}(x)).$$

"Pre-norm" means LayerNorm is applied to the input of each sub-layer (inside the residual), which trains more stably than the original "post-norm." The attention sub-layer mixes information *across* positions; the MLP sub-layer processes *each* position independently. Both wrapped in residuals so the block is easy to skip.

### Line by line

```python
self.attn = SelfAttention(ni, attn_chans=attn_chans, transpose=False)
```
The notebook-27 multi-head self-attention. `transpose=False` because our sequence is already in `(batch, seq, channels)` layout here — no need for the internal `(n,c,s)↔(n,s,c)` swap the conv-side code used.

```python
emb = self.emb_proj(F.silu(t))[:, None]
scale, shift = torch.chunk(emb, 2, dim=2)
```
FiLM conditioning, exactly as in notebook 28's `EmbResBlock`: project the time embedding `t` to `2·ni` numbers, split into a per-channel `scale` and `shift`. The `[:, None]` inserts a sequence axis so the `(batch, 1, 2ni)` embedding **broadcasts across all sequence positions** — the time signal is the same for every position (it describes the whole image's noise level), which is exactly right.

```python
x = x + self.attn(self.nrm1(x))          # residual pre-norm attention
```
Sub-layer 1: LayerNorm, self-attention, residual add. Positions exchange information.

```python
x = x*(1+scale) + shift                   # FiLM time modulation
```
Inject the noise-level condition between the two sub-layers via `x*(1+scale)+shift`. As in notebook 28, it is `1+scale` (not `scale`) so that at init (`emb_proj≈0`) the modulation is the identity and does not disrupt the feature scale — the network learns how strongly to modulate.

```python
return x + self.mlp(self.nrm2(x))         # residual pre-norm MLP
```
Sub-layer 2: LayerNorm, feed-forward MLP, residual add. Each position is processed nonlinearly.

### The upshot

`EmbTransformerBlk` = **standard transformer block + a FiLM modulation from the diffusion timestep.** Stacking `n_mids` of these at the U-Net bottleneck gives the network a small but full Transformer that reasons *globally* over the 64 bottleneck positions, conditioned on how noisy the image currently is. It is the same three ingredients you have already met — attention (nb 27), residual pre-norm structure (nb 13/transformers), and FiLM conditioning (nb 28) — assembled into the transformer form.


---

# Part 5: The Transformer-Bottleneck U-Net (`SimpleDiffusion`)

Now we assemble the full model: a convolutional encoder, a **transformer bottleneck** (the `LearnEmbSS` positional embedding + a stack of `EmbTransformerBlk`s), and a convolutional decoder — with the notebook-28 skip connections tying encoder to decoder.

---


In [ ]:
class SimpleDiffusion(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, nfs=(224,448,672,896), num_layers=1,
                 attn_chans=8, attn_start=1, n_mids=8):
        super().__init__()
        self.conv_in = nn.Conv2d(in_channels, nfs[0], kernel_size=3, padding=1)
        self.n_temb = nf = nfs[0]
        n_emb = nf*4
        self.emb_mlp = nn.Sequential(lin(self.n_temb, n_emb, norm=nn.BatchNorm1d),
                                     lin(n_emb, n_emb))
        self.downs = nn.ModuleList()
        n = len(nfs)
        for i in range(n):
            ni = nf
            nf = nfs[i]
            self.downs.append(DownBlock(n_emb, ni, nf, add_down=i!=n-1, num_layers=num_layers,
                                        attn_chans=0 if i<attn_start else attn_chans))

        self.le = LearnEmbSS(64, nf)
        self.mids = nn.ModuleList([EmbTransformerBlk(n_emb, nf) for _ in range(n_mids)])

        rev_nfs = list(reversed(nfs))
        nf = rev_nfs[0]
        self.ups = nn.ModuleList()
        for i in range(n):
            prev_nf = nf
            nf = rev_nfs[i]
            ni = rev_nfs[min(i+1, len(nfs)-1)]
            self.ups.append(UpBlock(n_emb, ni, prev_nf, nf, add_up=i!=n-1, num_layers=num_layers+1,
                                    attn_chans=0 if i>=n-attn_start else attn_chans))
        self.conv_out = pre_conv(nfs[0], out_channels, act=nn.SiLU, norm=nn.BatchNorm2d, bias=False)

    def forward(self, inp):
        x,t = inp
        temb = timestep_embedding(t, self.n_temb)
        emb = self.emb_mlp(temb)
        x = self.conv_in(x)
        saved = [x]
        for block in self.downs: x = block(x, emb)
        saved += [p for o in self.downs for p in o.saved]
        n,c,h,w = x.shape
        x = self.le(x.reshape(n,c,-1).transpose(1,2))
        for block in self.mids: x = block(x, emb)
        x = x.transpose(1,2).reshape(n,c,h,w)
        for block in self.ups: x = block(x, emb, saved)
        return self.conv_out(x)

**What does the code above do?**

The transformer-bottleneck U-Net. Structurally it is notebook 28's `EmbUNetModel` with the **conv mid-block swapped for a transformer stack.** The deep dive traces the forward pass; the constructor is the familiar down/up-block scaffolding plus the new `self.le` (positional embedding) and `self.mids` (the transformer blocks).


## Deep Dive: The Convolution → Transformer → Convolution Architecture

`SimpleDiffusion` is a **hybrid**: convolutions on the outside, a Transformer in the middle. This is a deliberate division of labor, and the `forward` pass shows exactly where one hands off to the other.

### Reading the forward pass

```python
temb = timestep_embedding(t, self.n_temb)   # scalar noise level -> sinusoidal vector (nb 28)
emb = self.emb_mlp(temb)                     # -> processed time embedding fed to every block
x = self.conv_in(x)                          # lift 3 RGB channels -> nfs[0] feature channels
saved = [x]
for block in self.downs: x = block(x, emb)   # CONV ENCODER: downsample, collect skips
saved += [p for o in self.downs for p in o.saved]
```

The **convolutional encoder** runs first, downsampling the 64×64 image through the `DownBlock`s to a small bottleneck feature map (with `nfs=(32,256,384,512)` and 3 downsamples, `64 → 32 → 16 → 8`, so the bottleneck is **8×8**), collecting skip connections along the way (the notebook-28 `saved()` mechanism).

```python
n,c,h,w = x.shape                            # bottleneck is (n, c, 8, 8)
x = self.le(x.reshape(n,c,-1).transpose(1,2))  # flatten to (n, 64, c) + add positional embedding
for block in self.mids: x = block(x, emb)    # TRANSFORMER BOTTLENECK: n_mids blocks
x = x.transpose(1,2).reshape(n,c,h,w)        # reshape back to (n, c, 8, 8)
```

The **transformer bottleneck** is the heart of the notebook. The 8×8 feature map is flattened to a **sequence of 64 positions** (the same flatten-to-sequence move as notebook 27), the `LearnEmbSS` positional embedding is applied so the transformer knows *where* each position sits, and then `n_mids=8` `EmbTransformerBlk`s process the sequence — global all-pairs attention over the 64 positions, plus per-position MLPs, all conditioned on the timestep. Afterward the sequence is reshaped back into an 8×8 map.

```python
for block in self.ups: x = block(x, emb, saved)   # CONV DECODER: upsample, consume skips
return self.conv_out(x)                            # -> 3 RGB noise-prediction channels
```

The **convolutional decoder** upsamples back to 64×64, consuming the skip connections, and `conv_out` projects to the 3-channel noise prediction.

### Why this division of labor

- **Convolutions on the outside (high resolution).** At 64×64 and 32×32, feature maps are large; attention's `O(S²)` cost would be prohibitive, and *local* operations (edges, textures) are exactly what convolutions excel at, cheaply and with translation equivariance. So the encoder/decoder stay convolutional.
- **Transformer in the middle (low resolution).** At the 8×8 bottleneck the sequence is only 64 positions — attention is cheap — and this is precisely where you want *global* reasoning: "does the overall layout of this bedroom cohere?" A stack of transformer blocks is far more expressive for that global mixing than a couple of conv layers.

This "conv stem → transformer core → conv head" pattern is not a diffusion quirk — it is the dominant design for applying Transformers to high-resolution grids where full-resolution attention is unaffordable, and (as noted in the intro) it is the skeleton of several production ML weather models. `SimpleDiffusion` is a clean, minimal instance of it.


---

# Part 6: Initialization and Training

`init_ddpm` (from the latent notebook / a diffusion-training standard) zero-initializes residual outputs so the U-Net starts near-identity. Then standard training on 64×64 pixels.

---


In [ ]:
def init_ddpm(model):
    for o in model.downs:
        for p in o.resnets: p.conv2[-1].weight.data.zero_()

    for o in model.ups:
        for p in o.resnets: p.conv2[-1].weight.data.zero_()

**What does the code above do?**

Zero-initializes the final layer of the second conv in every down/up residual block, so each block starts as an identity and the whole U-Net begins as a near-identity function — the diffusion-training stabilizer explained in detail in the latent notebook (notebook 30-latents). It touches only the *convolutional* down/up blocks; the transformer mids rely on their own residual + FiLM-`1+scale` + `LearnEmbSS`-zero-init soft start. Note this cell **re-defines** `init_ddpm` locally (it is also in `miniai.diffusion`); the local copy is identical.


In [ ]:
lr = 1e-3
epochs = 15
opt_func = partial(optim.AdamW, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched), MixedPrecision()]
model = SimpleDiffusion(in_channels=3, out_channels=3, nfs=(32,256,384,512), num_layers=1, attn_chans=0, n_mids=8)
init_ddpm(model)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

**What does the code above do?**

Assembles the training run:

| Component | Choice | Note |
|-----------|--------|------|
| Model | `SimpleDiffusion(nfs=(32,256,384,512), n_mids=8)` | 3→3 channels (RGB), 8 transformer mid-blocks |
| `attn_chans=0` | **conv blocks use no attention** here | all the global reasoning is delegated to the transformer bottleneck |
| Init | `init_ddpm(model)` | near-identity start |
| Optimizer | AdamW, `eps=1e-5` | |
| Schedule | OneCycle, `max_lr=1e-3`, 15 epochs | lower LR than the latent run |
| Loss | `MSELoss` | predict the noise `ε` |

Note `attn_chans=0`: the convolutional down/up blocks run *without* their optional in-block attention, because this architecture concentrates *all* attention in the transformer mids. The jump `nfs=(32,256,...)` (a big channel increase after the first block) gives the network capacity quickly while keeping the first, highest-resolution stage cheap.


In [ ]:
learn.fit(epochs)

**What does the code above do?**

Trains for 15 epochs. **What you should see:** MSE loss (predicted vs. true noise) descending. Pixel-space training at 64×64 is heavier per step than the latent notebook's, but the model is smaller and the resolution modest, so it remains tractable. (Plot/metrics omitted.)


In [ ]:
torch.save(learn.model.state_dict(), 'models/lsun_diffusion-prog-64.pkl')

**What does the code above do?**

Saves the trained weights to `models/lsun_diffusion-prog-64.pkl` so you can reload without retraining. (Unlike the latent notebook, this save is *not* commented out — it runs.) The `prog-64` in the name hints at a progressive-growing / 64×64 stage in a larger pipeline.


---

# Part 7: Sampling in Pixel Space

DDIM sampling, reusing notebook 28's `sample` loop with a slightly tweaked `ddim_step`. Since we diffuse in pixel space, the generated tensors *are* images — no VAE decode needed, just add back the 0.5 we subtracted.

---


In [ ]:
sz = (16,3,64,64)

**What does the code above do?**

The shape to generate: 16 RGB 64×64 images. Because this is pixel-space diffusion, the sampling target is a full image tensor (contrast the latent notebook, which generated 4×32×32 latents).


In [ ]:
def ddim_step(x_t, noise, abar_t, abar_t1, bbar_t, bbar_t1, eta, sig, clamp=1.):
    sig = ((bbar_t1/bbar_t).sqrt() * (1-abar_t/abar_t1).sqrt()) * eta
    x_0_hat = (x_t-(1-abar_t).sqrt()*noise  )   / abar_t.sqrt()
    if clamp: x_0_hat = x_0_hat.clamp(-clamp,clamp)
    if bbar_t1<=sig**2+0.01: sig=0.  # set to zero if very small or NaN
    x_t = abar_t1.sqrt()*x_0_hat + (bbar_t1-sig**2).sqrt()*noise
    x_t += sig * torch.randn(x_t.shape).to(x_t)
    return x_0_hat,x_t

**What does the code above do?**

The DDIM update from notebook 28, with one change: **`clamp` is now a numeric bound, not a boolean.** In notebook 28 `clamp=True` clamped the clean-image estimate to `[-1, 1]`; here `clamp=1.` clamps `x_0_hat` to `[-clamp, clamp] = [-1, 1]`, and you could pass a different bound. Since our images are centered to `[-0.5, 0.5]` the true data lies within `[-0.5, 0.5]`, so clamping to `[-1, 1]` is a loose, safe guard against runaway estimates without cutting into real signal. Everything else — recover `x_0_hat` by inverting the forward equation, compute the DDIM variance `sigma` (scaled by `eta`), re-noise to the next level — is exactly the notebook-28 `ddim_step`. (This cell re-defines `ddim_step` locally, shadowing the `miniai.diffusion` version, just to adjust the `clamp` semantics.)


In [ ]:
# set_seed(42)
preds = sample(ddim_step, model, sz, steps=100, eta=1., clamp=1.)

**What does the code above do?**

Runs notebook 28's DDIM `sample` loop for 100 steps (`eta=1`, stochastic), passing the local `ddim_step` with `clamp=1.`. It starts from pure noise and iteratively denoises 16 images. `preds[-1]` is the final generated batch (in the centered `[-0.5, 0.5]` convention). **What you should see:** a progress bar over 100 steps. (HTML progress output only.)


In [ ]:
s = (preds[-1]+0.5)
# s = preds[-1]
s.min(),s.max(),s.shape

(tensor(-0.0786), tensor(1.0906), torch.Size([16, 3, 64, 64]))

**What does the code above do?**

Takes the final generated images and **adds 0.5** to undo the centering, converting from `[-0.5, 0.5]` back to displayable `[0, 1]` pixels. The printed min/max (`≈ -0.08, 1.09`) show the values land near `[0, 1]` but slightly outside — hence the `clamp(0,1)` when displaying next. Shape is `(16, 3, 64, 64)`, our 16 generated RGB images.


In [ ]:
show_images(s[:9].clamp(0,1), imsize=2)

**What does the code above do?**

Displays 9 of the generated images, clamped to `[0, 1]`. **What you should see:** **novel 64×64 bedroom images** generated from noise — recognizable beds, windows, and walls, softer and lower-resolution than the latent notebook's 256×256 outputs (because we diffused directly at 64×64), but coherent bedrooms produced by the conv–transformer–conv model. (Image omitted.)

That completes the pixel-space pipeline: noise → 100 DDIM steps through the transformer-bottleneck U-Net → a 64×64 bedroom, no VAE required.


---

# Summary and What's Next

### What we built

| Piece | Role |
|-------|------|
| 64×64 pixel data | `avg_pool2d(im, 4)`, centered to `[-0.5, 0.5]` — cheap pixel-space diffusion |
| `LearnEmbSS` | Learned positional embedding (per-position scale+shift) for the bottleneck sequence |
| `EmbTransformerBlk` | Pre-norm transformer block (attention + MLP) with FiLM time conditioning |
| `SimpleDiffusion` | Conv encoder → **transformer bottleneck** → conv decoder U-Net |
| `init_ddpm` | Zero-init residual outputs → near-identity start |
| DDIM sampling | Pixel-space generation; `clamp` as a numeric bound; `+0.5` to display |

### The ideas to remember

1. **Two ways to make diffusion affordable.** Compress to a latent (the sibling notebook, Stable-Diffusion-style) *or* diffuse at low resolution in pixel space (this notebook). This one is the simplest working image generator — no VAE to manage.
2. **Transformer bottleneck.** Convolutions on the outside (cheap, local, high-res), a Transformer in the middle (global reasoning where the sequence is short and cheap). The `conv → transformer → conv` hybrid is the dominant pattern for Transformers on high-res grids — and the skeleton of modern ML weather models.
3. **Transformers need positions.** Attention is permutation-invariant, so a flattened image bottleneck needs a positional embedding; `LearnEmbSS` supplies a learned, zero-initialized one.
4. **`EmbTransformerBlk` = standard pre-norm transformer block + FiLM.** Attention (nb 27) + residual pre-norm structure + timestep FiLM (nb 28), assembled.
5. **Same diffusion core throughout.** `noisify`, `timestep_embedding`, `sample`, `ddim_step`, `init_ddpm`, `DownBlock`/`UpBlock` all come from notebook 28 unchanged; only the *bottleneck architecture* changed. The diffusion machinery is truly reusable.

### Where this goes

Notebook 31 (`imgnet_latents`) returns to the **latent** route and adds **class conditioning** for controllable generation on ImageNet — the last piece of the course's generative arc. The transformer-bottleneck idea here generalizes directly: swap the conv stem/head for patch-embed/unpatch and let the transformer core dominate, and you have a **Diffusion Transformer (DiT)** — the architecture behind the newest large text-to-image and video models. And the same conv→transformer→conv skeleton, pointed at atmospheric fields with a physical-forcing condition instead of a timestep, is a generative weather emulator.

### Suggested next steps

1. Re-read the three deep dives (`LearnEmbSS`, `EmbTransformerBlk`, the conv→transformer→conv architecture) — the transformer-block and bottleneck material is the transferable core.
2. Contrast this notebook with its latent sibling (`30_lsun_diffusion-latents`): same diffusion core, different way of making high-res affordable (low-res pixels + transformer bottleneck here vs. VAE latents there). Make sure you can articulate the trade-off.
3. When satisfied, run `concept-extraction` (candidates: low-res pixel-space diffusion, learned positional embedding / `LearnEmbSS`, pre-norm transformer block + FiLM, conv→transformer→conv bottleneck, DDIM `clamp` as numeric bound, centering/`+0.5`).
4. Optionally `/colab` for a GPU-ready version and `/html` to publish.

---
